In [ ]:
import csv
import pickle
import numpy as np
from skimage import io
from scipy.ndimage import binary_fill_holes

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================
TARGET_dataset = "min_234"
MIN_CELL_AREA = 10000

DATASET_CONFIGS = {
    1: {
        "image_path": "Lysosome_masked_crops_1024.tif",
        "csv_path": "PhTau_histogram_clusters.csv",
        "coord_path": "processed_data/spot_coordinates.pkl",
        "delaunay_path": "processed_data/delaunay_results.pkl",
    },
    2: {
        "image_path": "cell_gallery_1024x1024_6ch.tif",
        "csv_path": "gallery_phtau_ground_truth.csv",
        "coord_path": "processed_data_2/spot_coordinates.pkl",
        "delaunay_path": "processed_data_2/delaunay_results.pkl",
    },
}

# Master data containers
images = {}
n_images = {}
labels = {}
categories = {}
coord_map = {}
delaunay_objects_dict = {}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def load_csv_labels(csv_path, total_expected_frames):
    """Parses cluster labels from CSV and pads remaining frames as untreated."""
    parsed_labels = []
    with open(csv_path, "r") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header row
        for row in reader:
            val = row[-1]
            if val == "0":
                parsed_labels.append(["0", "dox+, PhTau+"])
            elif val == "1":
                parsed_labels.append(["1", "dox+, PhTau-"])

    # Pad missing indices up to the total stack size (assumed untreated)
    while len(parsed_labels) < total_expected_frames:
        parsed_labels.append(["2", "dox-"])
        
    return parsed_labels


def load_pickle_dataset(pickle_path, dataset_key):
    """Helper to extract a specific resolution dataset from a pickle file."""
    with open(pickle_path, "rb") as f:
        return pickle.load(f)[dataset_key]


def compute_valid_mask_indices(image_stack, area_threshold):
    """ Vectorised masking and area filter over a standardised image stack."""
    # Since channel is guaranteed to be at axis -1, collapse it if 4D
    raw_masks = np.any(image_stack > 0, axis=-1) if image_stack.ndim == 4 else image_stack > 0

    # Fill internal segmentation holes slice-by-slice across the stack
    cell_masks = binary_fill_holes(raw_masks)
    
    # Calculate pixel surface areas by summing spatial dimensions (axes 1 and 2)
    cell_areas = np.sum(cell_masks, axis=(1, 2))
    
    # Return boolean array masking frames that meet size criteria
    return cell_areas >= area_threshold

# ==========================================
# 3. PIPELINE EXECUTION
# ==========================================
for idx, config in DATASET_CONFIGS.items():
    # Load image stack pipeline
    raw_images_stack = io.imread(config["image_path"])

    # Standardise 4D stacks to always be (frames, spatial_1, spatial_2, channels)
    if raw_images_stack.ndim == 4:
        # If your channel is currently at axis 1 (e.g., shape 100, 6, 1024, 1024)
        channel_axis = np.argmin(raw_images_stack.shape)
        if channel_axis != -1:
            raw_images_stack = np.moveaxis(raw_images_stack, channel_axis, -1)
    
    total_frames = raw_images_stack.shape[0]

    # Load and map structural metadata sequentially
    all_labels = load_csv_labels(config["csv_path"], total_frames)
    full_coord = load_pickle_dataset(config["coord_path"], TARGET_dataset)
    full_delaunay = load_pickle_dataset(config["delaunay_path"], TARGET_dataset)

    # Compute valid mask filter indices using vectorised area sums
    valid_indices = compute_valid_mask_indices(raw_images_stack, MIN_CELL_AREA)

    # Identify the rejected indices where the condition is False
    rejected_indices = np.where(~valid_indices)[0]
    
    # Print the rejected cell indices for the current dataset
    print(f"Dataset {idx} ({config['image_path']}):")
    if len(rejected_indices) > 0:
        print(f"  Rejected {len(rejected_indices)} cells due to small area (< {MIN_CELL_AREA} px).")
        print(f"  Rejected Indices: {rejected_indices.tolist()}\n")
    else:
        print("  No cells were rejected.\n")

    # Apply boolean index filtering to numerical arrays
    images[idx] = raw_images_stack[valid_indices]
    n_images[idx] = np.sum(valid_indices)
    
    # Filter Python list metadata using matching list comprehensions
    labels[idx] = [all_labels[i] for i, valid in enumerate(valid_indices) if valid]
    coord_map[idx] = [full_coord[i] for i, valid in enumerate(valid_indices) if valid]
    delaunay_objects_dict[idx] = [full_delaunay[i] for i, valid in enumerate(valid_indices) if valid]

    # Parse remaining categories, sorting by category id descending
    unique_pairs = {tuple(item) for item in labels[idx]}
    sorted_pairs = sorted(unique_pairs, key=lambda x: int(x[0]), reverse=True)
    categories[idx] = [pair[1] for pair in sorted_pairs]

In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay

# Select the first retained Delaunay object from Dataset 1
first_delaunay = delaunay_objects_dict[1][0]

# Plot the Delaunay triangulation
fig, ax = plt.subplots(figsize=(4, 4))

if isinstance(first_delaunay, Delaunay):
    points = first_delaunay.points

    ax.triplot(
        points[:, 0],
        points[:, 1],
        first_delaunay.simplices,
        color="black",
        linewidth=0.6,
        alpha=0.8,
    )
    ax.scatter(
        points[:, 0],
        points[:, 1],
        s=10,
        color="red",
        zorder=2,
    )

elif hasattr(first_delaunay, "nodes") and hasattr(first_delaunay, "edges"):
    # Support NetworkX graph objects
    positions = nx.get_node_attributes(first_delaunay, "pos")

    nx.draw(
        first_delaunay,
        pos=positions,
        ax=ax,
        node_size=10,
        node_color="red",
        edge_color="black",
        width=0.6,
    )

else:
    raise TypeError(
        f"Unsupported Delaunay object type: {type(first_delaunay)}"
    )

ax.set_title("Delaunay graph")
ax.set_aspect("equal")
ax.invert_yaxis()  # Match image-coordinate convention

# Remove the numbers/ticks and the outer box
ax.axis("off")

plt.tight_layout()
plt.savefig("Figures/delaunay_graph.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
from types import SimpleNamespace

filtered_delaunay_dict = {}
all_edge_lengths = []

# Step 1: Collect edge lengths across all datasets
for data_idx, delaunay_list in delaunay_objects_dict.items():
    # Fetch the corresponding flat list of coordinates
    points_list = coord_map[data_idx]

    # Loop directly through the items using a single index
    for i, delaunay_obj in enumerate(delaunay_list):
        if delaunay_obj is None:
            continue
            
        simplices = delaunay_obj.simplices
        if simplices.shape[0] > 0:
            # Match the Delaunay object directly with its coordinate array
            pts = np.array(points_list[i])

            # Extract vertices for the first edge of each triangle
            v0, v1 = pts[simplices[:, 0]], pts[simplices[:, 1]]
            distances = np.linalg.norm(v0 - v1, axis=1)
            all_edge_lengths.extend(distances.tolist())

# Step 2: Calculate global threshold using the IQR method
all_edge_lengths = np.array(all_edge_lengths)
q3 = np.percentile(all_edge_lengths, 75)
iqr = q3 - np.percentile(all_edge_lengths, 25)
max_edge_length = q3 + 1.5 * iqr

print(f"Automatic filtering threshold: {max_edge_length:.2f} pixels")

# Step 3: Filter and wrap back into a flat list structure
for data_idx, delaunay_list in delaunay_objects_dict.items():
    points_list = coord_map[data_idx]
    filtered_list = []

    for i, delaunay_obj in enumerate(delaunay_list):
        if delaunay_obj is None:
            filtered_list.append(None)
            continue
            
        simplices = delaunay_obj.simplices
        pts = np.array(points_list[i])

        if simplices.shape[0] == 0:
            filtered_list.append(SimpleNamespace(simplices=simplices))
            continue

        v0 = pts[simplices[:, 0]]
        v1 = pts[simplices[:, 1]]
        v2 = pts[simplices[:, 2]]

        d01 = np.linalg.norm(v0 - v1, axis=1)
        d12 = np.linalg.norm(v1 - v2, axis=1)
        d20 = np.linalg.norm(v2 - v0, axis=1)

        mask = (
            (d01 <= max_edge_length)
            & (d12 <= max_edge_length)
            & (d20 <= max_edge_length)
        )

        filtered_obj = SimpleNamespace(simplices=simplices[mask])
        filtered_list.append(filtered_obj)

    # Stores a flat list matching your new data structure format
    filtered_delaunay_dict[data_idx] = filtered_list

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. STYLE AND SETTINGS
# ==========================================

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

pixel_width = 0.0739859
area_conversion = pixel_width ** 2

full_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

# Accept both possible label formats
label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    return label_map.get(label, label)


# ==========================================
# 2. COMBINE DATASETS
# ==========================================

coord_map["combined"] = coord_map[1] + coord_map[2]

filtered_delaunay_dict["combined"] = (
    filtered_delaunay_dict[1]
    + filtered_delaunay_dict[2]
)

labels["combined"] = labels[1] + labels[2]

datasets = [1, 2, "combined"]


# ==========================================
# 3. COMPUTE TRIANGLE AREAS AND STATISTICS
# ==========================================

def compute_area(points, tri):
    triangles = points[tri.simplices]
    a, b, c = triangles[:, 0], triangles[:, 1], triangles[:, 2]

    return 0.5 * np.abs(
        (b[:, 0] - a[:, 0]) * (c[:, 1] - a[:, 1])
        - (b[:, 1] - a[:, 1]) * (c[:, 0] - a[:, 0])
    )


triangle_areas = {}

for dataset in datasets:
    triangle_areas[dataset] = []

    for im_id in range(len(coord_map[dataset])):
        points = np.asarray(coord_map[dataset][im_id])
        tri = filtered_delaunay_dict[dataset][im_id]

        if tri is None or len(points) < 3:
            triangle_areas[dataset].append(np.array([]))
            continue

        triangle_areas[dataset].append(compute_area(points, tri))


area_stats = {
    dataset: {
        # Convert pixel areas to µm²
        "mean": np.array([
            np.mean(areas) * area_conversion
            if len(areas) else np.nan
            for areas in triangle_areas[dataset]
        ]),
        "median": np.array([
            np.median(areas) * area_conversion
            if len(areas) else np.nan
            for areas in triangle_areas[dataset]
        ]),
        "n_tri": np.array([
            len(areas)
            for areas in triangle_areas[dataset]
        ])
    }
    for dataset in datasets
}


# ==========================================
# 4. PREPARE DATAFRAMES
# ==========================================

def make_dataframe(dataset):
    condition_names = [
        standardise_condition(label[1])
        for label in labels[dataset]
    ]

    return pd.DataFrame({
        "Mean_triangle_area_um2": area_stats[dataset]["mean"],
        "Median_triangle_area_um2": area_stats[dataset]["median"],
        "Number_of_triangles": area_stats[dataset]["n_tri"],
        "Condition": condition_names
    }).dropna()


df1 = make_dataframe(1)
df2 = make_dataframe(2)
df_combined = make_dataframe("combined")


# ==========================================
# 5. PLOTTING FUNCTION
# ==========================================

plot_definitions = [
    (
        "Mean_triangle_area_um2",
        "Mean Triangle Areas",
        "Mean triangle area (µm²)",
        "mean_triangle_area"
    ),
    (
        "Median_triangle_area_um2",
        "Median Triangle Areas",
        "Median triangle area (µm²)",
        "median_triangle_area"
    ),
    (
        "Number_of_triangles",
        "Number of Triangles",
        "Number of triangles",
        "number_of_triangles"
    )
]


def create_violin_plots(
    column,
    title,
    ylabel,
    filename_prefix
):
    # --------------------------------------
    # Plot 1: Dataset 1 and Dataset 2
    # --------------------------------------
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 3),
        sharey=True
    )

    datasets_for_plot = [
        (
            df1,
            [
                "dox-",
                "dox+\nPhTau-",
                "dox+\nPhTau+"
            ],
            "Dataset 1",
            axes[0]
        ),
        (
            df2,
            [
                "dox+\nPhTau-",
                "dox+\nPhTau+"
            ],
            "Dataset 2",
            axes[1]
        )
    ]

    for df_subset, order_subset, dataset_title, ax in datasets_for_plot:
        sns.violinplot(
            ax=ax,
            data=df_subset,
            x="Condition",
            y=column,
            hue="Condition",
            order=order_subset,
            hue_order=full_order,
            palette=traffic_light_map,
            legend=False,
            cut=1
        )

        ax.set_title(
            f"{title} ({dataset_title})",
            fontsize=14,
            pad=10
        )
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_xlabel("Condition", fontsize=12)
        ax.grid(axis="y", linestyle="--", alpha=0.5)

        plt.sca(ax)
        plt.xticks(rotation=0)

    plt.savefig(
        f"Figures/{filename_prefix}_by_dataset.svg",
        format="svg",
        bbox_inches="tight"
    )
    plt.show()


    # --------------------------------------
    # Plot 2: Combined Dataset
    # --------------------------------------
    fig, ax = plt.subplots(figsize=(5, 3))

    sns.violinplot(
        ax=ax,
        data=df_combined,
        x="Condition",
        y=column,
        hue="Condition",
        order=full_order,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )

    ax.set_title(
        title,
        fontsize=14,
        pad=10
    )
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.xticks(rotation=0)

    plt.savefig(
        f"Figures/{filename_prefix}_combined.svg",
        format="svg",
        bbox_inches="tight"
    )
    plt.show()


# ==========================================
# 6. CREATE ALL FIGURES
# ==========================================

for column, title, ylabel, filename_prefix in plot_definitions:
    create_violin_plots(
        column=column,
        title=title,
        ylabel=ylabel,
        filename_prefix=filename_prefix
    )


# ==========================================
# 7. SUMMARY STATISTICS
# ==========================================

def print_summary(df, dataset_name):
    print(f"\n--- {dataset_name} Summary Statistics ---")

    print(
        df.groupby("Condition")[
            [
                "Mean_triangle_area_um2",
                "Median_triangle_area_um2",
                "Number_of_triangles"
            ]
        ]
        .describe()
        .round(2)
    )


print_summary(df1, "Dataset 1")
print_summary(df2, "Dataset 2")
print_summary(df_combined, "Combined Dataset")

In [ ]:
from itertools import combinations

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests


def run_group_tests(
    df,
    value_col,
    dataset_name,
    metric_name,
    condition_order,
    group_col="Condition",
    alpha=0.05
):
    """
    Run a Kruskal–Wallis test followed by pairwise Mann–Whitney U tests.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame.
    value_col : str
        Numeric measurement column.
    dataset_name : str
        Dataset name printed in the output.
    metric_name : str
        Human-readable measurement name.
    condition_order : list
        Desired order of groups.
    group_col : str, default="Condition"
        Column containing the group labels.
    alpha : float, default=0.05
        Significance threshold.
    """

    # Validate required columns
    missing_columns = [
        column for column in [value_col, group_col]
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Missing column(s): {missing_columns}. "
            f"Available columns are: {list(df.columns)}"
        )

    print(f"\n{'=' * 75}")
    print(f"{metric_name} — {dataset_name}")
    print(f"{'=' * 75}")

    # Extract non-empty groups in the requested order
    groups = {
        group: df.loc[
            df[group_col] == group,
            value_col
        ].dropna()
        for group in condition_order
    }

    # Remove groups with no observations
    groups = {
        group: values
        for group, values in groups.items()
        if len(values) > 0
    }

    print("\nSample sizes:")
    for group, values in groups.items():
        print(f"  {group}: n = {len(values)}")

    if len(groups) < 2:
        print("\nNot enough groups for statistical testing.")
        return None

    # Kruskal–Wallis omnibus test
    kw_stat, kw_p = kruskal(*groups.values())

    print("\nKruskal–Wallis test:")
    print(f"  H = {kw_stat:.4f}")
    print(f"  p = {kw_p:.4g}")

    if kw_p < alpha:
        print(f"  Significant overall difference at alpha = {alpha}.")
    else:
        print(f"  No significant overall difference at alpha = {alpha}.")

    # Pairwise Mann–Whitney U tests
    pairwise_results = []

    for group_a, group_b in combinations(groups.keys(), 2):
        values_a = groups[group_a]
        values_b = groups[group_b]

        u_stat, raw_p = mannwhitneyu(
            values_a,
            values_b,
            alternative="two-sided",
            method="asymptotic"
        )

        pairwise_results.append({
            "Metric": metric_name,
            "Dataset": dataset_name,
            "Group column": group_col,
            "Comparison": f"{group_a} vs {group_b}",
            "U": u_stat,
            "Raw_p": raw_p
        })

    # Holm correction for multiple comparisons
    raw_p_values = [
        result["Raw_p"]
        for result in pairwise_results
    ]

    _, adjusted_p_values, _, _ = multipletests(
        raw_p_values,
        method="holm",
        alpha=alpha
    )

    for result, adjusted_p in zip(
        pairwise_results,
        adjusted_p_values
    ):
        result["Holm_adjusted_p"] = adjusted_p
        result["Significant"] = adjusted_p < alpha

    results_df = pd.DataFrame(pairwise_results)

    print("\nPairwise Mann–Whitney U tests with Holm correction:")
    print(
        results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    return results_df

# Statistical analysis settings
metric_definitions = [
    (
        "Mean_triangle_area_um2",
        "Mean triangle area (µm²)"
    ),
    (
        "Median_triangle_area_um2",
        "Median triangle area (µm²)"
    ),
    (
        "Number_of_triangles",
        "Number of triangles"
    )
]

dataset_definitions = [
    (df1, "Dataset 1"),
    (df2, "Dataset 2"),
    (df_combined, "Combined Dataset")
]

all_pairwise_results = []

for df, dataset_name in dataset_definitions:
    for value_col, metric_name in metric_definitions:

        results = run_group_tests(
            df=df,
            value_col=value_col,
            dataset_name=dataset_name,
            metric_name=metric_name,
            condition_order=full_order,
            group_col="Condition",
            alpha=0.05
        )

        if results is not None and not results.empty:
            all_pairwise_results.append(results)

# Combine all pairwise results into one table
if all_pairwise_results:
    all_pairwise_results_df = pd.concat(
        all_pairwise_results,
        ignore_index=True
    )

    print("\n\nComplete pairwise-results table:")
    print(
        all_pairwise_results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    # Optional: save results
    all_pairwise_results_df.to_csv(
        "triangle_area_statistical_results.csv",
        index=False
    )
else:
    all_pairwise_results_df = pd.DataFrame()
    print("No pairwise results were generated.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. STYLE AND SETTINGS
# ==========================================

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

pixel_width = 0.0739859
area_conversion = pixel_width ** 2

order_1 = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_2 = [
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

full_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

# Support both comma-separated and newline-separated labels
label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    return label_map.get(label, label)


# ==========================================
# 2. PREPARE RAW TRIANGLE-AREA DATA
# ==========================================

def make_raw_area_dataframe(dataset):
    rows = []

    for image_index, label in enumerate(labels[dataset]):
        condition = standardise_condition(label[1])

        # Individual triangle areas for this image
        areas = triangle_areas[dataset][image_index]

        if len(areas) == 0:
            continue

        for area in areas:
            rows.append({
                "Area_um2": area * area_conversion,
                "Condition": condition
            })

    return pd.DataFrame(rows).dropna()


df1 = make_raw_area_dataframe(1)
df2 = make_raw_area_dataframe(2)
df_combined = make_raw_area_dataframe("combined")


# ==========================================
# 3. DATASET 1 AND DATASET 2 SIDE-BY-SIDE
# ==========================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 3),
    sharey=True
)

datasets_for_plot = [
    (df1, order_1, "Individual Triangle Areas (Dataset 1)", axes[0]),
    (df2, order_2, "Individual Triangle Areas (Dataset 2)", axes[1])
]

for df_subset, order_subset, title_text, ax in datasets_for_plot:
    sns.violinplot(
        ax=ax,
        data=df_subset,
        x="Condition",
        y="Area_um2",
        hue="Condition",
        order=order_subset,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )

    ax.set_title(title_text, fontsize=14, pad=10)
    ax.set_ylabel("Triangle area (µm²)", fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.sca(ax)
    plt.xticks(rotation=0)

plt.savefig(
    "Figures/raw_triangle_area_by_dataset.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()


# ==========================================
# 4. COMBINED DATASET PLOT
# ==========================================

fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_combined,
    x="Condition",
    y="Area_um2",
    hue="Condition",
    order=full_order,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=1
)

ax.set_title(
    "Individual Triangle Areas",
    fontsize=14,
    pad=10
)
ax.set_ylabel("Triangle area (µm²)", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.xticks(rotation=0)

plt.savefig(
    "Figures/raw_triangle_area_combined.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()


# ==========================================
# 5. SUMMARY STATISTICS
# ==========================================

print("--- Dataset 1 Summary Statistics (µm²) ---")
print(
    df1.groupby("Condition")["Area_um2"]
    .describe()
    .round(2)
)

print("\n--- Dataset 2 Summary Statistics (µm²) ---")
print(
    df2.groupby("Condition")["Area_um2"]
    .describe()
    .round(2)
)

print("\n--- Combined Dataset Summary Statistics (µm²) ---")
print(
    df_combined.groupby("Condition")["Area_um2"]
    .describe()
    .round(2)
)

In [ ]:
# ==========================================
# 6. STATISTICAL ANALYSIS
# ==========================================

raw_area_results = []

dataset_definitions = [
    (df1, "Dataset 1", order_1),
    (df2, "Dataset 2", order_2),
    (df_combined, "Combined Dataset", full_order)
]

for df, dataset_name, condition_order in dataset_definitions:

    results = run_group_tests(
        df=df,
        value_col="Area_um2",
        dataset_name=dataset_name,
        metric_name="Individual triangle area (µm²)",
        condition_order=condition_order,
        group_col="Condition",
        alpha=0.05
    )

    if results is not None and not results.empty:
        raw_area_results.append(results)


# Combine all pairwise results
if raw_area_results:
    raw_area_results_df = pd.concat(
        raw_area_results,
        ignore_index=True
    )

    print("\n\nCombined pairwise results:")
    print(
        raw_area_results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    raw_area_results_df.to_csv(
        "triangle_area_raw_statistical_results.csv",
        index=False
    )
else:
    raw_area_results_df = pd.DataFrame()
    print("No pairwise results were generated.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. STYLE AND SETTINGS
# ==========================================

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

pixel_width = 0.0739859

order_1 = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_2 = [
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

full_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    return label_map.get(label, label)


# ==========================================
# 2. COMPUTE UNIQUE DELAUNAY EDGE LENGTHS
# ==========================================

def compute_edge_lengths(points, tri):
    edges = np.concatenate([
        tri.simplices[:, [0, 1]],
        tri.simplices[:, [1, 2]],
        tri.simplices[:, [2, 0]]
    ])

    # Sort endpoints so that edge direction does not matter
    edges = np.sort(edges, axis=1)

    # Keep each unique edge only once
    unique_edges = np.unique(edges, axis=0)

    p1 = points[unique_edges[:, 0]]
    p2 = points[unique_edges[:, 1]]

    return np.linalg.norm(p1 - p2, axis=1)


edge_lengths_all = {
    dataset: []
    for dataset in datasets
}

for dataset in datasets:
    for im_id in range(len(coord_map[dataset])):
        points = np.asarray(coord_map[dataset][im_id])
        tri = filtered_delaunay_dict[dataset][im_id]

        if tri is None or len(points) < 2:
            edge_lengths_all[dataset].append(np.array([]))
            continue

        edge_lengths_all[dataset].append(
            compute_edge_lengths(points, tri)
        )


edge_stats = {
    dataset: {
        # Convert pixel distances to µm
        "mean": np.array([
            lengths.mean() * pixel_width
            if len(lengths) else np.nan
            for lengths in edge_lengths_all[dataset]
        ]),
        "median": np.array([
            np.median(lengths) * pixel_width
            if len(lengths) else np.nan
            for lengths in edge_lengths_all[dataset]
        ]),
        "n_edges": np.array([
            len(lengths)
            for lengths in edge_lengths_all[dataset]
        ])
    }
    for dataset in datasets
}


# ==========================================
# 3. PREPARE IMAGE-LEVEL DATAFRAMES
# ==========================================

def make_dataframe(dataset):
    condition_names = [
        standardise_condition(label[1])
        for label in labels[dataset]
    ]

    return pd.DataFrame({
        "Mean_edge_length_um": edge_stats[dataset]["mean"],
        "Median_edge_length_um": edge_stats[dataset]["median"],
        "Number_of_edges": edge_stats[dataset]["n_edges"],
        "Condition": condition_names
    }).dropna()


df1 = make_dataframe(1)
df2 = make_dataframe(2)
df_combined = make_dataframe("combined")


# ==========================================
# 4. VIOLIN-PLOT FUNCTION
# ==========================================

plot_definitions = [
    (
        "Mean_edge_length_um",
        "Mean Edge Lengths",
        "Mean edge length (µm)",
        "mean_edge_length"
    ),
    (
        "Median_edge_length_um",
        "Median Edge Lengths",
        "Median edge length (µm)",
        "median_edge_length"
    ),
    (
        "Number_of_edges",
        "Number of Edges",
        "Number of edges",
        "number_of_edges"
    )
]


def create_violin_plots(
    column,
    title,
    ylabel,
    filename_prefix
):
    # --------------------------------------
    # Dataset 1 and Dataset 2 side-by-side
    # --------------------------------------
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 3),
        sharey=True
    )

    datasets_for_plot = [
        (
            df1,
            order_1,
            f"{title} (Dataset 1)",
            axes[0]
        ),
        (
            df2,
            order_2,
            f"{title} (Dataset 2)",
            axes[1]
        )
    ]

    for df_subset, order_subset, title_text, ax in datasets_for_plot:
        sns.violinplot(
            ax=ax,
            data=df_subset,
            x="Condition",
            y=column,
            hue="Condition",
            order=order_subset,
            hue_order=full_order,
            palette=traffic_light_map,
            legend=False,
            cut=1
        )

        ax.set_title(title_text, fontsize=14, pad=10)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_xlabel("Condition", fontsize=12)
        ax.grid(axis="y", linestyle="--", alpha=0.5)

        plt.sca(ax)
        plt.xticks(rotation=0)

    plt.savefig(
        f"Figures/{filename_prefix}_by_dataset.svg",
        format="svg",
        bbox_inches="tight"
    )
    plt.show()


    # --------------------------------------
    # Combined dataset
    # --------------------------------------
    fig, ax = plt.subplots(figsize=(5, 3))

    sns.violinplot(
        ax=ax,
        data=df_combined,
        x="Condition",
        y=column,
        hue="Condition",
        order=full_order,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )

    ax.set_title(title, fontsize=14, pad=10)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.xticks(rotation=0)

    plt.savefig(
        f"Figures/{filename_prefix}_combined.svg",
        format="svg",
        bbox_inches="tight"
    )
    plt.show()


# ==========================================
# 5. CREATE ALL PLOTS
# ==========================================

for column, title, ylabel, filename_prefix in plot_definitions:
    create_violin_plots(
        column=column,
        title=title,
        ylabel=ylabel,
        filename_prefix=filename_prefix
    )


# ==========================================
# 6. SUMMARY STATISTICS
# ==========================================

def print_summary(df, dataset_name):
    print(f"\n--- {dataset_name} Summary Statistics ---")

    print(
        df.groupby("Condition")[
            [
                "Mean_edge_length_um",
                "Median_edge_length_um",
                "Number_of_edges"
            ]
        ]
        .describe()
        .round(2)
    )


print_summary(df1, "Dataset 1")
print_summary(df2, "Dataset 2")
print_summary(df_combined, "Combined Dataset")

In [ ]:
# ==========================================
# 7. STATISTICAL ANALYSIS
# ==========================================

edge_metric_definitions = [
    (
        "Mean_edge_length_um",
        "Mean edge length (µm)"
    ),
    (
        "Median_edge_length_um",
        "Median edge length (µm)"
    ),
    (
        "Number_of_edges",
        "Number of edges"
    )
]

edge_dataset_definitions = [
    (df1, "Dataset 1", order_1),
    (df2, "Dataset 2", order_2),
    (df_combined, "Combined Dataset", full_order)
]

edge_pairwise_results = []

for df, dataset_name, condition_order in edge_dataset_definitions:

    for value_col, metric_name in edge_metric_definitions:

        results = run_group_tests(
            df=df,
            value_col=value_col,
            dataset_name=dataset_name,
            metric_name=metric_name,
            condition_order=condition_order,
            group_col="Condition",
            alpha=0.05
        )

        if results is not None and not results.empty:
            edge_pairwise_results.append(results)


# Combine all pairwise results
if edge_pairwise_results:

    edge_pairwise_results_df = pd.concat(
        edge_pairwise_results,
        ignore_index=True
    )

    print("\n\nComplete edge-metric pairwise results:")
    print(
        edge_pairwise_results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    edge_pairwise_results_df.to_csv(
        "edge_length_statistical_results.csv",
        index=False
    )

else:
    edge_pairwise_results_df = pd.DataFrame()
    print("No pairwise results were generated.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. STYLE AND SETTINGS
# ==========================================

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

pixel_width = 0.0739859

order_1 = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_2 = [
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

full_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    return label_map.get(label, label)


# ==========================================
# 2. PREPARE RAW EDGE-LENGTH DATA
# ==========================================

def make_raw_edge_dataframe(dataset):
    rows = []

    for image_index, label in enumerate(labels[dataset]):
        condition = standardise_condition(label[1])

        # Individual edge lengths for this image
        lengths = edge_lengths_all[dataset][image_index]

        if len(lengths) == 0:
            continue

        for length in lengths:
            rows.append({
                # Convert pixel distances to µm
                "Edge_length_um": length * pixel_width,
                "Condition": condition
            })

    return pd.DataFrame(rows).dropna()


df1 = make_raw_edge_dataframe(1)
df2 = make_raw_edge_dataframe(2)
df_combined = make_raw_edge_dataframe("combined")


# ==========================================
# 3. DATASET 1 AND DATASET 2 SIDE-BY-SIDE
# ==========================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 3),
    sharey=True
)

datasets_for_plot = [
    (
        df1,
        order_1,
        "Individual Edge Lengths (Dataset 1)",
        axes[0]
    ),
    (
        df2,
        order_2,
        "Individual Edge Lengths (Dataset 2)",
        axes[1]
    )
]

for df_subset, order_subset, title_text, ax in datasets_for_plot:
    sns.violinplot(
        ax=ax,
        data=df_subset,
        x="Condition",
        y="Edge_length_um",
        hue="Condition",
        order=order_subset,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )

    ax.set_title(title_text, fontsize=14, pad=10)
    ax.set_ylabel("Edge length (µm)", fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.sca(ax)
    plt.xticks(rotation=0)

plt.savefig(
    "Figures/raw_edge_lengths_by_dataset.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()


# ==========================================
# 4. COMBINED DATASET PLOT
# ==========================================

fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_combined,
    x="Condition",
    y="Edge_length_um",
    hue="Condition",
    order=full_order,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=1
)

ax.set_title(
    "Individual Edge Lengths",
    fontsize=14,
    pad=10
)
ax.set_ylabel("Edge length (µm)", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.xticks(rotation=0)

plt.savefig(
    "Figures/raw_edge_lengths_combined.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()


# ==========================================
# 5. SUMMARY STATISTICS
# ==========================================

print("--- Dataset 1 Summary Statistics (µm) ---")
print(
    df1.groupby("Condition")["Edge_length_um"]
    .describe()
    .round(2)
)

print("\n--- Dataset 2 Summary Statistics (µm) ---")
print(
    df2.groupby("Condition")["Edge_length_um"]
    .describe()
    .round(2)
)

print("\n--- Combined Dataset Summary Statistics (µm) ---")
print(
    df_combined.groupby("Condition")["Edge_length_um"]
    .describe()
    .round(2)
)

In [ ]:
# ==========================================
# 6. STATISTICAL ANALYSIS
# ==========================================

raw_edge_results = []

dataset_definitions = [
    (df1, "Dataset 1", order_1),
    (df2, "Dataset 2", order_2),
    (df_combined, "Combined Dataset", full_order)
]

for df, dataset_name, condition_order in dataset_definitions:

    results = run_group_tests(
        df=df,
        value_col="Edge_length_um",
        dataset_name=dataset_name,
        metric_name="Individual edge length (µm)",
        condition_order=condition_order,
        group_col="Condition",
        alpha=0.05
    )

    if results is not None and not results.empty:
        raw_edge_results.append(results)


# Combine and save pairwise results
if raw_edge_results:

    raw_edge_results_df = pd.concat(
        raw_edge_results,
        ignore_index=True
    )

    print("\n\nComplete raw edge-length pairwise results:")
    print(
        raw_edge_results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    raw_edge_results_df.to_csv(
        "raw_edge_length_statistical_results.csv",
        index=False
    )

else:
    raw_edge_results_df = pd.DataFrame()
    print("No pairwise results were generated.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. STYLE AND SETTINGS
# ==========================================

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

order_1 = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_2 = [
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

full_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    return label_map.get(label, label)


# ==========================================
# 2. COMPUTE TRIANGLE ASPECT RATIOS
# ==========================================

def compute_shape_metrics(points, tri):
    triangles = points[tri.simplices]

    d0 = np.linalg.norm(
        triangles[:, 0] - triangles[:, 1],
        axis=1
    )
    d1 = np.linalg.norm(
        triangles[:, 1] - triangles[:, 2],
        axis=1
    )
    d2 = np.linalg.norm(
        triangles[:, 2] - triangles[:, 0],
        axis=1
    )

    edges = np.stack([d0, d1, d2], axis=1)

    # Longest edge / shortest edge
    return np.max(edges, axis=1) / np.clip(
        np.min(edges, axis=1),
        1e-6,
        None
    )


shape_metrics = {
    dataset: []
    for dataset in datasets
}

for dataset in datasets:
    for im_id in range(len(coord_map[dataset])):
        points = np.asarray(coord_map[dataset][im_id])
        tri = filtered_delaunay_dict[dataset][im_id]

        if tri is None or len(points) < 3:
            shape_metrics[dataset].append(np.array([]))
            continue

        shape_metrics[dataset].append(
            compute_shape_metrics(points, tri)
        )


# ==========================================
# 3. CALCULATE IMAGE-LEVEL STATISTICS
# ==========================================

shape_stats = {
    dataset: {
        "Mean_aspect_ratio": np.array([
            ratios.mean()
            if len(ratios) else np.nan
            for ratios in shape_metrics[dataset]
        ]),
        "Median_aspect_ratio": np.array([
            np.median(ratios)
            if len(ratios) else np.nan
            for ratios in shape_metrics[dataset]
        ]),
        "Maximum_aspect_ratio": np.array([
            np.max(ratios)
            if len(ratios) else np.nan
            for ratios in shape_metrics[dataset]
        ])
    }
    for dataset in datasets
}


# ==========================================
# 4. PREPARE DATAFRAMES
# ==========================================

def make_dataframe(dataset):
    condition_names = [
        standardise_condition(label[1])
        for label in labels[dataset]
    ]

    return pd.DataFrame({
        "Mean_aspect_ratio": shape_stats[dataset]["Mean_aspect_ratio"],
        "Median_aspect_ratio": shape_stats[dataset]["Median_aspect_ratio"],
        "Maximum_aspect_ratio": shape_stats[dataset]["Maximum_aspect_ratio"],
        "Condition": condition_names
    }).dropna()


df1 = make_dataframe(1)
df2 = make_dataframe(2)
df_combined = make_dataframe("combined")


# ==========================================
# 5. VIOLIN-PLOT FUNCTION
# ==========================================

plot_definitions = [
    (
        "Mean_aspect_ratio",
        "Mean Aspect Ratio",
        "Longest edge / shortest edge",
        "mean_aspect_ratio"
    ),
    (
        "Median_aspect_ratio",
        "Median Aspect Ratio",
        "Longest edge / shortest edge",
        "median_aspect_ratio"
    ),
    (
        "Maximum_aspect_ratio",
        "Maximum Aspect Ratio",
        "Longest edge / shortest edge",
        "maximum_aspect_ratio"
    )
]


def create_violin_plots(
    column,
    title,
    ylabel,
    filename_prefix
):
    # --------------------------------------
    # Dataset 1 and Dataset 2 side-by-side
    # --------------------------------------
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 3),
        sharey=True
    )

    datasets_for_plot = [
        (
            df1,
            order_1,
            f"{title} (Dataset 1)",
            axes[0]
        ),
        (
            df2,
            order_2,
            f"{title} (Dataset 2)",
            axes[1]
        )
    ]

    for df_subset, order_subset, title_text, ax in datasets_for_plot:
        sns.violinplot(
            ax=ax,
            data=df_subset,
            x="Condition",
            y=column,
            hue="Condition",
            order=order_subset,
            hue_order=full_order,
            palette=traffic_light_map,
            legend=False,
            cut=1
        )

        ax.set_title(title_text, fontsize=14, pad=10)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_xlabel("Condition", fontsize=12)
        ax.grid(axis="y", linestyle="--", alpha=0.5)

        plt.sca(ax)
        plt.xticks(rotation=0)

    plt.savefig(
        f"Figures/{filename_prefix}_by_dataset.svg",
        format="svg",
        bbox_inches="tight"
    )
    plt.show()


    # --------------------------------------
    # Combined dataset
    # --------------------------------------
    fig, ax = plt.subplots(figsize=(5, 3))

    sns.violinplot(
        ax=ax,
        data=df_combined,
        x="Condition",
        y=column,
        hue="Condition",
        order=full_order,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )

    ax.set_title(title, fontsize=14, pad=10)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.xticks(rotation=0)

    plt.savefig(
        f"Figures/{filename_prefix}_combined.svg",
        format="svg",
        bbox_inches="tight"
    )
    plt.show()


# ==========================================
# 6. CREATE ALL PLOTS
# ==========================================

for column, title, ylabel, filename_prefix in plot_definitions:
    create_violin_plots(
        column=column,
        title=title,
        ylabel=ylabel,
        filename_prefix=filename_prefix
    )


# ==========================================
# 7. SUMMARY STATISTICS
# ==========================================

def print_summary(df, dataset_name):
    print(f"\n--- {dataset_name} Summary Statistics ---")

    print(
        df.groupby("Condition")[
            [
                "Mean_aspect_ratio",
                "Median_aspect_ratio",
                "Maximum_aspect_ratio"
            ]
        ]
        .describe()
        .round(2)
    )


print_summary(df1, "Dataset 1")
print_summary(df2, "Dataset 2")
print_summary(df_combined, "Combined Dataset")

In [ ]:
# ==========================================
# 8. STATISTICAL ANALYSIS
# ==========================================

aspect_metric_definitions = [
    (
        "Mean_aspect_ratio",
        "Mean triangle aspect ratio"
    ),
    (
        "Median_aspect_ratio",
        "Median triangle aspect ratio"
    ),
    (
        "Maximum_aspect_ratio",
        "Maximum triangle aspect ratio"
    )
]

aspect_dataset_definitions = [
    (df1, "Dataset 1", order_1),
    (df2, "Dataset 2", order_2),
    (df_combined, "Combined Dataset", full_order)
]

aspect_pairwise_results = []

for df, dataset_name, condition_order in aspect_dataset_definitions:

    for value_col, metric_name in aspect_metric_definitions:

        results = run_group_tests(
            df=df,
            value_col=value_col,
            dataset_name=dataset_name,
            metric_name=metric_name,
            condition_order=condition_order,
            group_col="Condition",
            alpha=0.05
        )

        if results is not None and not results.empty:
            aspect_pairwise_results.append(results)


# Combine all pairwise results
if aspect_pairwise_results:

    aspect_pairwise_results_df = pd.concat(
        aspect_pairwise_results,
        ignore_index=True
    )

    print("\n\nComplete aspect-ratio pairwise results:")
    print(
        aspect_pairwise_results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    aspect_pairwise_results_df.to_csv(
        "aspect_ratio_statistical_results.csv",
        index=False
    )

else:
    aspect_pairwise_results_df = pd.DataFrame()
    print("No pairwise results were generated.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. STYLE AND SETTINGS
# ==========================================

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

order_1 = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_2 = [
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

full_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    return label_map.get(label, label)


# ==========================================
# 2. PREPARE RAW TRIANGLE-SHAPE DATA
# ==========================================

def make_raw_shape_dataframe(dataset):
    rows = []

    for image_index, label in enumerate(labels[dataset]):
        condition = standardise_condition(label[1])

        # Individual triangle aspect ratios for this image
        aspect_ratios = shape_metrics[dataset][image_index]

        if len(aspect_ratios) == 0:
            continue

        for ratio in aspect_ratios:
            rows.append({
                "Aspect_ratio": ratio,
                "Condition": condition
            })

    return pd.DataFrame(rows).dropna()


df1 = make_raw_shape_dataframe(1)
df2 = make_raw_shape_dataframe(2)
df_combined = make_raw_shape_dataframe("combined")


# ==========================================
# 3. DATASET 1 AND DATASET 2 SIDE-BY-SIDE
# ==========================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 3),
    sharey=True
)

datasets_for_plot = [
    (
        df1,
        order_1,
        "Individual Triangle Shapes (Dataset 1)",
        axes[0]
    ),
    (
        df2,
        order_2,
        "Individual Triangle Shapes (Dataset 2)",
        axes[1]
    )
]

for df_subset, order_subset, title_text, ax in datasets_for_plot:
    sns.violinplot(
        ax=ax,
        data=df_subset,
        x="Condition",
        y="Aspect_ratio",
        hue="Condition",
        order=order_subset,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )

    ax.set_title(title_text, fontsize=14, pad=10)
    ax.set_ylabel(
        "Aspect ratio\n(longest edge / shortest edge)",
        fontsize=12
    )
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    plt.sca(ax)
    plt.xticks(rotation=0)

plt.savefig(
    "Figures/raw_triangle_shapes_by_dataset.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()


# ==========================================
# 4. COMBINED DATASET PLOT
# ==========================================

fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_combined,
    x="Condition",
    y="Aspect_ratio",
    hue="Condition",
    order=full_order,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=1
)

ax.set_title(
    "Individual Triangle Shapes",
    fontsize=14,
    pad=10
)
ax.set_ylabel(
    "Aspect ratio\n(longest edge / shortest edge)",
    fontsize=12
)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.xticks(rotation=0)

plt.savefig(
    "Figures/raw_triangle_shapes_combined.svg",
    format="svg",
    bbox_inches="tight"
)
plt.show()


# ==========================================
# 5. SUMMARY STATISTICS
# ==========================================

print("--- Dataset 1 Summary Statistics ---")
print(
    df1.groupby("Condition")["Aspect_ratio"]
    .describe()
    .round(2)
)

print("\n--- Dataset 2 Summary Statistics ---")
print(
    df2.groupby("Condition")["Aspect_ratio"]
    .describe()
    .round(2)
)

print("\n--- Combined Dataset Summary Statistics ---")
print(
    df_combined.groupby("Condition")["Aspect_ratio"]
    .describe()
    .round(2)
)

In [ ]:
# ==========================================
# 6. STATISTICAL ANALYSIS
# ==========================================

raw_shape_results = []

dataset_definitions = [
    (df1, "Dataset 1", order_1),
    (df2, "Dataset 2", order_2),
    (df_combined, "Combined Dataset", full_order)
]

for df, dataset_name, condition_order in dataset_definitions:

    results = run_group_tests(
        df=df,
        value_col="Aspect_ratio",
        dataset_name=dataset_name,
        metric_name="Individual triangle aspect ratio",
        condition_order=condition_order,
        group_col="Condition",
        alpha=0.05
    )

    if results is not None and not results.empty:
        raw_shape_results.append(results)


# Combine and save pairwise results
if raw_shape_results:

    raw_shape_results_df = pd.concat(
        raw_shape_results,
        ignore_index=True
    )

    print("\n\nComplete raw aspect-ratio pairwise results:")
    print(
        raw_shape_results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    raw_shape_results_df.to_csv(
        "raw_triangle_aspect_ratio_statistical_results.csv",
        index=False
    )

else:
    raw_shape_results_df = pd.DataFrame()
    print("No pairwise results were generated.")

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. FEATURE DEFINITIONS
# ==========================================

datasets_to_process = [1, 2, "combined"]

feature_names = [
    "Mean_triangle_area_um2",
    "Median_triangle_area_um2",
    "Number_of_triangles",
    "Mean_edge_length_um",
    "Median_edge_length_um",
    "Number_of_edges",
    "Mean_aspect_ratio",
    "Median_aspect_ratio",
    "Maximum_aspect_ratio"
]


# ==========================================
# 2. ENSURE COMBINED STATISTICS EXIST
# ==========================================

def combine_statistics(statistics_dict):
    if "combined" not in statistics_dict:
        statistics_dict["combined"] = {
            key: np.concatenate([
                statistics_dict[1][key],
                statistics_dict[2][key]
            ])
            for key in statistics_dict[1]
        }


combine_statistics(area_stats)
combine_statistics(edge_stats)
combine_statistics(shape_stats)


# ==========================================
# 3. HANDLE ASPECT-RATIO KEY NAMES
# ==========================================

# The adapted shape-metrics cell uses these keys.
# The fallback keys support the original naming convention.
shape_key_map = {
    "Mean_aspect_ratio": [
        "Mean_aspect_ratio",
        "mean_aspect"
    ],
    "Median_aspect_ratio": [
        "Median_aspect_ratio",
        "median_aspect"
    ],
    "Maximum_aspect_ratio": [
        "Maximum_aspect_ratio",
        "max_aspect"
    ]
}


def get_shape_stat(dataset, feature_name):
    for key in shape_key_map[feature_name]:
        if key in shape_stats[dataset]:
            return shape_stats[dataset][key]

    raise KeyError(
        f"Could not find a matching key for {feature_name}. "
        f"Available keys are: {list(shape_stats[dataset].keys())}"
    )


# ==========================================
# 4. BUILD IMAGE-LEVEL FEATURE MATRICES
# ==========================================

feature_matrices = {}
feature_dataframes = {}

for ds_key in datasets_to_process:
    n_images = len(labels[ds_key])
    rows = []

    mean_aspect = get_shape_stat(ds_key, "Mean_aspect_ratio")
    median_aspect = get_shape_stat(ds_key, "Median_aspect_ratio")
    maximum_aspect = get_shape_stat(ds_key, "Maximum_aspect_ratio")

    for image_index in range(n_images):
        condition = labels[ds_key][image_index][1]

        rows.append({
            "Image_index": image_index,
            "Condition": condition,

            "Mean_triangle_area_um2": (
                area_stats[ds_key]["mean"][image_index]
            ),
            "Median_triangle_area_um2": (
                area_stats[ds_key]["median"][image_index]
            ),
            "Number_of_triangles": (
                area_stats[ds_key]["n_tri"][image_index]
            ),

            "Mean_edge_length_um": (
                edge_stats[ds_key]["mean"][image_index]
            ),
            "Median_edge_length_um": (
                edge_stats[ds_key]["median"][image_index]
            ),
            "Number_of_edges": (
                edge_stats[ds_key]["n_edges"][image_index]
            ),

            "Mean_aspect_ratio": mean_aspect[image_index],
            "Median_aspect_ratio": median_aspect[image_index],
            "Maximum_aspect_ratio": maximum_aspect[image_index]
        })

    feature_df = pd.DataFrame(rows)

    # Remove images lacking any required geometric feature
    feature_df = feature_df.dropna(
        subset=feature_names
    ).reset_index(drop=True)

    feature_dataframes[ds_key] = feature_df

    feature_matrices[ds_key] = feature_df[
        feature_names
    ].to_numpy(dtype=float)


# ==========================================
# 5. CHECK OUTPUTS
# ==========================================

print("Feature order:")
for index, feature_name in enumerate(feature_names):
    print(f"{index}: {feature_name}")

for ds_key in datasets_to_process:
    dataset_name = (
        "Combined Dataset"
        if ds_key == "combined"
        else f"Dataset {ds_key}"
    )

    print(
        f"\n{dataset_name} feature matrix shape: "
        f"{feature_matrices[ds_key].shape}"
    )

    print(feature_dataframes[ds_key].head())


# ==========================================
# 6. OPTIONAL: SAVE MATRICES
# ==========================================

for ds_key in datasets_to_process:
    dataset_name = (
        "combined"
        if ds_key == "combined"
        else f"dataset_{ds_key}"
    )

    feature_dataframes[ds_key].to_csv(
        f"Figures/{dataset_name}_geometric_feature_matrix.csv",
        index=False
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

datasets_to_plot = [1, 2, "combined"]


def minmax_normalise_columns(matrix):
    """
    Normalise each feature column independently to the range 0–1.
    Constant-valued columns are assigned zero.
    """
    matrix = np.asarray(matrix, dtype=float)

    column_min = np.nanmin(matrix, axis=0)
    column_max = np.nanmax(matrix, axis=0)
    column_range = column_max - column_min

    normalised = np.zeros_like(matrix, dtype=float)

    non_constant = column_range > 0

    normalised[:, non_constant] = (
        matrix[:, non_constant] - column_min[non_constant]
    ) / column_range[non_constant]

    return normalised


def plot_feature_matrix(ds_key):
    """
    Plot a column-normalised geometric feature matrix heatmap.
    """

    if (
        ds_key not in feature_matrices
        or feature_matrices[ds_key].size == 0
    ):
        print(f"No matrix data available for Dataset/View: {ds_key}")
        return

    matrix = feature_matrices[ds_key]

    # Normalise each feature independently for visualisation
    matrix_normalised = minmax_normalise_columns(matrix)

    dataset_name = (
        "Combined Datasets"
        if ds_key == "combined"
        else f"Dataset {ds_key}"
    )

    fig, ax = plt.subplots(figsize=(12, 8))

    sns.heatmap(
        matrix_normalised,
        ax=ax,
        cmap="viridis",
        vmin=0,
        vmax=1,
        cbar_kws={
            "label": "Column-normalised feature value"
        },
        xticklabels=feature_names,
        yticklabels=False
    )

    ax.set_title(
        f"Geometric Feature Matrix Heatmap — {dataset_name}",
        fontsize=14,
        pad=10
    )

    ax.set_xlabel("Geometric features", fontsize=12)
    ax.set_ylabel("Image index", fontsize=12)

    plt.xticks(
        rotation=45,
        ha="right",
        fontsize=10
    )

    plt.savefig(
        f"Figures/geometric_feature_matrix_{ds_key}.svg",
        format="svg",
        bbox_inches="tight"
    )

    plt.show()


# ==========================================
# CREATE HEATMAPS
# ==========================================

for ds_key in datasets_to_plot:
    plot_feature_matrix(ds_key)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ==========================================
# 1. SETTINGS
# ==========================================

traffic_light_map = {
    "dox-": "#2ecc71",
    "dox+\nPhTau-": "#f1c40f",
    "dox+\nPhTau+": "#e74c3c"
}

canonical_order = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_1 = [
    "dox-",
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

order_2 = [
    "dox+\nPhTau-",
    "dox+\nPhTau+"
]

datasets_to_plot = [1, 2, "combined"]


# ==========================================
# 2. STANDARDISE CONDITION LABELS
# ==========================================

label_map = {
    "dox-": "dox-",
    "dox+, PhTau-": "dox+\nPhTau-",
    "dox+, PhTau+": "dox+\nPhTau+",
    "dox+\nPhTau-": "dox+\nPhTau-",
    "dox+\nPhTau+": "dox+\nPhTau+"
}


def standardise_condition(label):
    return label_map.get(label, label)


# Ensure the metadata labels use the same condition names
for ds_key in datasets_to_plot:
    if ds_key in feature_dataframes:
        feature_dataframes[ds_key]["Condition"] = (
            feature_dataframes[ds_key]["Condition"]
            .map(standardise_condition)
        )


# ==========================================
# 3. FIT PCA ON THE COMBINED FEATURE MATRIX
# ==========================================

if (
    "combined" not in feature_matrices
    or feature_matrices["combined"].size == 0
):
    raise ValueError(
        "The combined feature matrix is empty and PCA cannot be fitted."
    )

X_combined = np.asarray(
    feature_matrices["combined"],
    dtype=float
)

if X_combined.shape[0] < 2:
    raise ValueError(
        "At least two valid combined images are required for PCA."
    )

# Standardise because the features have different units and scales
scaler = StandardScaler()
X_combined_scaled = scaler.fit_transform(X_combined)

# Fit PCA once on the combined feature matrix
pca = PCA(n_components=2)
X_combined_pca = pca.fit_transform(X_combined_scaled)


# ==========================================
# 4. TRANSFORM ALL DATASETS USING SAME PCA
# ==========================================

pca_coordinates = {
    "combined": X_combined_pca
}

for ds_key in [1, 2]:
    if (
        ds_key not in feature_matrices
        or feature_matrices[ds_key].size == 0
    ):
        pca_coordinates[ds_key] = np.empty((0, 2))
        continue

    X_dataset = np.asarray(
        feature_matrices[ds_key],
        dtype=float
    )

    X_dataset_scaled = scaler.transform(X_dataset)
    pca_coordinates[ds_key] = pca.transform(X_dataset_scaled)


# ==========================================
# 5. SAVE PCA LOADINGS
# ==========================================

pca_loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_names,
    columns=["PC1_loading", "PC2_loading"]
)

pca_loadings.to_csv(
    "Figures/geometric_feature_pca_loadings.csv"
)

print("--- PCA Explained Variance ---")
print(
    pd.DataFrame({
        "Principal component": ["PC1", "PC2"],
        "Explained variance ratio": pca.explained_variance_ratio_,
        "Explained variance (%)": (
            pca.explained_variance_ratio_ * 100
        )
    }).round(3)
)

print("\n--- PCA Loadings ---")
print(pca_loadings.round(3))


# ==========================================
# 6. PLOTTING FUNCTION
# ==========================================

def plot_pca_points(ax, ds_key):
    coordinates = pca_coordinates[ds_key]

    if coordinates.size == 0:
        ax.axis("off")
        return

    if ds_key not in feature_dataframes:
        ax.axis("off")
        return

    current_labels = feature_dataframes[ds_key]["Condition"].to_numpy()

    if len(current_labels) != len(coordinates):
        raise ValueError(
            f"Number of labels and PCA coordinates does not match "
            f"for dataset {ds_key}."
        )

    active_categories = (
        order_2
        if ds_key == 2
        else canonical_order
    )

    for condition in active_categories:
        mask = current_labels == condition

        if not np.any(mask):
            continue

        ax.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            label=condition,
            color=traffic_light_map[condition],
            alpha=0.8,
            edgecolors="white",
            linewidth=0.5,
            s=20
        )

    dataset_title = (
        None
        if ds_key == "combined"
        else f"Dataset {ds_key}"
    )

    ax.set_title(
        dataset_title,
        fontsize=14,
        pad=10
    )

    ax.set_xlabel(
        f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
        fontsize=12
    )

    ax.set_ylabel(
        f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
        fontsize=12
    )

    ax.grid(
        True,
        linestyle="--",
        alpha=0.5
    )

    ax.legend(
        frameon=True,
        fontsize=9
    )


# ==========================================
# 7. DATASET 1 AND DATASET 2 SIDE-BY-SIDE
# ==========================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 3),
    sharex=True,
    sharey=True
)

plot_pca_points(axes[0], 1)
plot_pca_points(axes[1], 2)

plt.savefig(
    "Figures/geometric_features_pca_by_dataset.svg",
    format="svg",
    bbox_inches="tight"
)

plt.show()


# ==========================================
# 8. SEPARATE COMBINED PCA PLOT
# ==========================================

fig, ax = plt.subplots(figsize=(5, 3))

plot_pca_points(ax, "combined")

plt.savefig(
    "Figures/geometric_features_pca_combined.svg",
    format="svg",
    bbox_inches="tight"
)

plt.show()